# Volleyball Group Activity Recognition - Setup & Configuration

This notebook implements and evaluates deep learning models to classify group activities in volleyball matches. 

### Phase 1: Environment Setup
In this section, we load the required libraries for numerical computation, data manipulation, and operating system / file path utilities.

---


In [1]:
# all import statements 
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import pathlib
from pathlib import Path 
import shutil
import os 
from matplotlib  import pyplot as plt 

In [2]:

# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## 2. Clone Project Repository

We clone the custom codebase from GitHub to access our baseline model definitions (`models/`), custom dataset loading scripts (`src/`), utility functions (`utils/`), and configuration profiles (`configs/`).

---


In [3]:
!git clone https://github.com/omarTBakr/Volleyball-Group-Activity-Recognition.git   


# !git pull https://github.com/omarTBakr/Volleyball-Group-Activity-Recognition.git # to force the clonning 

# to handle the confilct

# !git config pull.rebase false
# !git pull origin main --no-edit

# # # 1. Discard the local changes to the lock files
# !git restore pyproject.toml uv.lock   


# # 2. Pull the latest code from GitHub
# !git pull


Cloning into 'Volleyball-Group-Activity-Recognition'...
remote: Enumerating objects: 734, done.
remote: Counting objects: 100% (290/290), done.
remote: Compressing objects: 100% (215/215), done.
remote: Total 734 (delta 142), reused 213 (delta 71), pack-reused 444 (from 1)
Receiving objects: 100% (734/734), 25.80 MiB | 36.90 MiB/s, done.
Resolving deltas: 100% (364/364), done.


## 3. Set Working Directory

By default, Kaggle notebooks execute from `/kaggle/working/`. We change the active working directory to the root of our cloned repository. This allows Python to locate and import our custom modules (e.g. `import models` or `from utils import ...`) relative to the project root.

---


In [4]:
 # change the working directory 
os.chdir('/kaggle/working/Volleyball-Group-Activity-Recognition')
print(f"Current Directory: {os.getcwd()}")


Current Directory: /kaggle/working/Volleyball-Group-Activity-Recognition


## 4. Path Validation

We run a validation check on our project path configurations. This script ensures that all required directories (such as the raw dataset images, training logs, and checkpoint folders) exist and are mapped correctly to Kaggle's local filesystem structure before training begins.

---


In [5]:
# validate the paths in path_config.py
from configs.path_config import validate_paths
validate_paths()

  [✓] /kaggle/input/datasets/ahmedmohamed365/volleyball
  [✓] /kaggle/working/Volleyball-Group-Activity-Recognition/saved_models
  [✓] /kaggle/input/datasets/ahmedmohamed365/volleyball/volleyball_/videos
  [✓] /kaggle/input/datasets/ahmedmohamed365/volleyball/videos_sample/videos_sample
  [✓] /kaggle/input/datasets/ahmedmohamed365/volleyball/volleyball_/videos
  [✓] /kaggle/input/datasets/ahmedmohamed365/volleyball/volleyball-detections/volleyball-detections
  [✓] /kaggle/input/datasets/ahmedmohamed365/volleyball/volleyball_tracking_annotation/volleyball_tracking_annotation


## 5. Data Preprocessing & Serialization

To avoid disk-bound bottlenecks during training, we pre-process the annotations and metadata:
1. **`json_parser`**: Transforms raw, space-separated annotation files into a structured, unified JSON format.
2. **`pickle_dump`**: Serializes annotations and frame indexes into Python pickle files. This enables sub-millisecond reading times when batching data in the PyTorch `DataLoader`.

*(Note: We skip loading frames into LMDB on Kaggle to save memory overhead, reading directly from the serialized layout instead).*

---


In [6]:
 # parse the paths into json format for easy access  
 
!python -m src.json_parser
!python -m src.pickle_dump
#!python -m src.load_frames_into_lmdb #  no need for this , we will read directly , on kaggle but will be quite handy  

INFO:__main__:Master JSON saved to: /kaggle/working/Volleyball-Group-Activity-Recognition/DataSet/volleyball_master.json (4830 clips)
INFO:__main__:Collected scene labels for 4830 clips across 55 videos.
✅ Merged scene labels for 4830 clips.
INFO:__main__:Enriched master JSON saved to: /kaggle/working/Volleyball-Group-Activity-Recognition/DataSet/volleyball_master.json
Successfully dumped pickle to /kaggle/working/Volleyball-Group-Activity-Recognition/DataSet/volleyball_master_pickle.pkl


## 6. Install Additional Packages & Test DataLoader

Before proceeding to training, we install any supplementary utilities (like `wrapt`) and run a sanity check on our data loading pipeline. Running the data loader script directly verifies that images are parsed, transformed, and batched without errors.

---


In [7]:
!uv add wrapt
!uv run python -m src.data.kaggle_data_loader

Using CPython 3.12.12 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 115 packages in 15ms
⠙ Preparing packages... (0/112)
⠙ Preparing packages... (0/112)
⠙ Preparing packages... (0/112)
pyyaml               ------------------------------     0 B/788.94 KiB
⠙ Preparing packages... (0/112)
pyyaml               ------------------------------     0 B/788.94 KiB
⠙ Preparing packages... (0/112)
pyyaml               ------------------------------     0 B/788.94 KiB
⠙ Preparing packages... (0/112)
pydantic             ------------------------------     0 B/461.19 KiB
pyyaml               ------------------------------     0 B/788.94 KiB
⠙ Preparing packages... (0/112)
pydantic             ------------------------------     0 B/461.19 KiB
pyyaml               ------------------------------     0 B/788.94 KiB
⠙ Preparing packages... (0/112)
pydantic             ------------------------------     0 B/461.19 KiB
pyyaml               ------------------------------ 

## 7. Inspect Path Configurations

We execute the path configuration script standalone to print and inspect the dynamically resolved file paths. This ensures we can double-check the absolute directories for configs, inputs, and output runs in the current environment.

---


In [8]:
!uv run python -m  configs.path_config

<frozen runpy>:128: RuntimeWarning: 'configs.path_config' found in sys.modules after import of package 'configs', but prior to execution of 'configs.path_config'; this may result in unpredictable behaviour
=== path_config (Kaggle) ===
  Data source:       /kaggle/input/datasets/ahmedmohamed365/volleyball
  Main dataset:      /kaggle/input/datasets/ahmedmohamed365/volleyball/volleyball_/videos
  Model save dir:    /kaggle/working/Volleyball-Group-Activity-Recognition/saved_models
  Detections:        /kaggle/input/datasets/ahmedmohamed365/volleyball/volleyball-detections/volleyball-detections
  Tracking:          /kaggle/input/datasets/ahmedmohamed365/volleyball/volleyball_tracking_annotation/volleyball_tracking_annotation
  Master JSON:       /kaggle/working/Volleyball-Group-Activity-Recognition/DataSet/volleyball_master.json
  Pickle cache:      /kaggle/working/Volleyball-Group-Activity-Recognition/DataSet/volleyball_master_pickle.pkl
  LMDB frames:       /kaggle/working/Volleyball-Gr

## 8. TensorBoard Visualization

To monitor training curves, learning rate adjustments, and validation metrics in real time, we can run TensorBoard. We have two options for doing this in Kaggle:

### Option A: Inline Magic (Recommended for Safari/Chrome with Cookies Enabled)
Use the `%tensorboard` notebook extension to display the UI directly in the notebook output cell.

### Option B: LocalTunnel Exporter (Fallback for Firefox / Brave / Cookie-blocked Browsers)
Exposes the local TensorBoard port (6006) to a secure public URL so you can view the charts in a separate, full-screen tab. 

---


In [9]:
# does not work on firefox 
# %load_ext tensorboard
# %tensorboard --logdir logs/baseline1/tensorboard/ --port 6007

# creating a local tunnel - best to do after training or clone and run it locally 

# #uncomment for the first time use
#1. Install localtunnel 
# !npm install -g localtunnel

# # 2. Start tensorboard in the background on port 6006
# !tensorboard --logdir logs/baseline1/tensorboard/ --port 6006 &

# 3. Expose port 6006 to the internet
#!lt --port 6006


 

## 9. Train Baseline 1 Model

We run the main training script for Baseline 1. 

According to our configurations, this model:
1. Performs a **Stage 1 (Warmup)** training run to train only the classifier head for 5 epochs.
2. Performs a **Stage 2 (Fine-tuning)** run to update the entire backbone using differential learning rates for 100 epochs.
3. Automatically logs checkpoints, metrics, and parameters using Hydra and TensorBoard.

---


In [10]:
#! uv add hydra-core tensorboard wrapt

!uv run python -m models.baseline1


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|███████████████████████████████████████| 97.8M/97.8M [00:00<00:00, 209MB/s]

  STAGE 1: Linear Probe (10 epochs, lr=0.001)

--- Stage 1 · Epoch 1/10 ---
Validation: 100%|████████████████████████████| 21/21 [00:13<00:00,  1.56batch/s]
Train -> Loss: 2.0440, Acc: 0.1724, F1: 0.0877
Val   -> Loss: 2.0426, Acc: 0.1797, F1: 0.0863
  ✓ New best model saved (F1: 0.0863)

--- Stage 1 · Epoch 2/10 ---
Validation: 100%|████████████████████████████| 21/21 [00:11<00:00,  1.82batch/s]
Train -> Loss: 2.0256, Acc: 0.1840, F1: 0.0777
Val   -> Loss: 2.0376, Acc: 0.1827, F1: 0.0971
  ✓ New best model saved (F1: 0.0971)

--- Stage 1 · Epoch 3/10 ---
Validation: 100%|████████████████████████████| 21/21 [00:11<00:00,  1.75batch/s]
Train -> Loss: 2.0154, Acc: 0.1775, F1: 0.1029
Val   -> Loss: 2.0308, Acc: 0.1894, F1: 0.0991
  ✓ New best model saved (F1: 0.0991)

--- Stage